# CITRO-DATA LAB COLIMA — Laboratorio Reproducible E2
## Fuente Especializada Geoespacial, Linaje de Datos, Diseño de Experimentos ($2^3$, $2\times 3$, $2^{4-1}$) e ISO/IEC 25010–25012

* **Estudiante:** Norberto Daniel Escareño Montaño (`23460323` · `@NorbertoDaniel65`)
* **Asignaturas:** Ingeniería de Datos (`CMD-2602`) · Análisis Estadístico para Ciencia de Datos
* **Etapa:** `E2` (Tag `e2`)


## 1. Fuente Especializada Geoespacial (OpenStreetMap + INEGI) y Unión Espacial `GeoPandas`
- **Polígonos municipales de Colima (`EPSG:4326`):** `10` (`4` núcleos citrícolas + `6` vecinos)
- **Unión espacial `gpd.sjoin(puntos, poligonos, predicate='within')`:** `1164` evaluados $\to$ `1074` aprobados en Colima + `90` en cuarentena (0 borrados silenciosamente, `0` PII).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd

ROOT = Path('..').resolve() if Path.cwd().name in ('laboratorio', 'E2') else Path('.').resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

gdf_oro = gpd.read_file(ROOT / 'data/oro/perfil_geoespacial_municipios_oro.geojson')
cols = ['cvegeo','nombre_oficial','en_observatorio_citricola','superficie_km2',
        'n_huertas_citricolas_osm','n_empaque_bodega_osm','n_mercados_abasto_osm',
        'n_nodos_agrologisticos_total','dist_puerto_manzanillo_km','dist_corredor_gdl_54d_km']
gdf_oro[cols]

## 2. Linaje y Trazabilidad Completa de Datos (Capa Bronce $\to$ Capa Oro)
- **Cobertura verificada:** `100.0%` (`310` registros maestros en Capa Oro rastreables hasta archivo Bronce, `SHA-256` y fecha UTC).

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from pipeline.linaje_datos import rastrear_registro_oro

traza = rastrear_registro_oro('ORO-SERIE-2022-02')
print(json.dumps(traza['registro_oro'], indent=2, ensure_ascii=False))
print('Etapas DAG:', traza['cadena_etapas'])

## 3. Experimento Factorial Real $2^3$, ANOVA de Dos Vías ($2\times 3$) y $2^4$ vs. $2^{4-1}$ Fraccionado
- **Óptimo $2^3$:** `A+B+C+` (`4_Hilos_Paralelo + Pandera_Vectorizado + Parquet_Snappy`)
- **Puerta 3 (Supuestos de Residuos):** Shapiro-Wilk `W=0.9582` (`p=0.4026`), Levene `p=0.7674`, Durbin-Watson `DW=2.0277`.
- **Confirmación ($n=5$):** Observado `70.934 ms` dentro de IC 95% `[56.932, 82.895]` $\implies$ **`-78.95%` tiempo (`4.75×` speedup) y `-91.13%` disco.**

In [ ]:
anova_23 = pd.read_csv(ROOT / 'data/oro/anova_factorial_2_3_e2.csv')
efectos_23 = pd.read_csv(ROOT / 'data/oro/efectos_factorial_2_3_e2.csv')
anova_2x3 = pd.read_csv(ROOT / 'data/oro/anova_dos_vias_2x3_e2.csv')
alias_24 = pd.read_csv(ROOT / 'data/oro/estructura_alias_2_4_fraccionado_e2.csv')
print('=== ANOVA FACTORIAL 2^3 ===')
display(anova_23)
print('=== EFECTOS ESTANDARIZADOS 2^3 ===')
display(efectos_23)
print('=== ANOVA DOS VIAS 2x3 ===')
display(anova_2x3)
print('=== ESTRUCTURA DE ALIAS 2^(4-1) RESOLUCION IV ===')
display(alias_24)

## 4. Evaluación ISO/IEC 25010 (Producto) e ISO/IEC 25012 (Datos)
- **ISO/IEC 25010 promedio:** `95.38/100`
- **ISO/IEC 25012 promedio:** `97.58/100` (Completitud `100.0%`, Linaje `100.0%`, `0` secretos expuestos).

In [ ]:
iso_25010 = pd.read_csv(ROOT / 'data/oro/evaluacion_iso_25010_e2.csv')
iso_25012 = pd.read_csv(ROOT / 'data/oro/evaluacion_iso_25012_e2.csv')
display(iso_25010)
display(iso_25012)